In [0]:
df = spark.read.option("multiLine", "true").json("/Volumes/workspace/example/example/N06-24_HighwaySection.geojson")
df.display()


In [0]:
from pyspark.sql.functions import explode, col

features_df = (
    df
    .select(explode("features").alias("feature"))
    .select(
        col("feature.geometry.type").alias("geom_type"),
        col("feature.geometry.coordinates").alias("coordinates"),
        col("feature.properties.*")
    )
)

features_df.display()


In [0]:
features_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.example.HighwaySection"
)


In [0]:
from sedona.spark import SedonaContext

SedonaContext.create(spark)


In [0]:
from pyspark.sql.functions import expr

geo_df = features_df.withColumn(
    "geom",
    expr("ST_GeomFromGeoJSON(to_json(named_struct('type', geom_type, 'coordinates', coordinates)))")
)

geo_df.select("geom").display()


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW geojson_raw
AS SELECT * FROM json.`/Volumes/workspace/example/example/N06-24_Joint.geojson`;

CREATE OR REPLACE TEMP VIEW features AS
SELECT
  f.geometry.type AS geom_type,
  f.geometry.coordinates AS coordinates,
  f.properties.*
FROM geojson_raw
LATERAL VIEW explode(features) AS f;
